### Задание:
### Работа тех.поддержки предполагает выполнение api-запросов для ответа клиенту. Поэтому Вам предлагается попрактиковаться в этом на примере открытых api.
### Возьмите любое публичное API и покажите взаимодействие пользователя для получения информации через API, с использованием OpenAI.

### Можете использовать публичное API для получения прогноза погоды: https://openweathermap.org/forecast5 (прогноз на 5 дней с интервалом 3 часа)
### Для этого зарегистрируйтесь и получите ключ здесь: https://home.openweathermap.org/users/sign_up

### Условие: Пользователь задает вопрос через модель OpenAI одним запросом с указанием всей интересующей его информации. Информация для ответа должна быть получена через API.
Наример, если текущая дата 2024-08-21 14:00, вопрос: Дай прогноз погоды в Москве на послезавтра вечером.
### Возможный формат ответа от OpenAI:
Город: Москва  
Прогноз на 2024-08-23 18:00:00  
Температура: 22.73 C° пасмурно  
Влажность: 66 %  
Давление: 1016 мм.рт.ст.  
Ветер: 4.1 м/с


### Код

In [1]:
#@title Библиотеки
!pip install -q openai==1.59.9

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.5/455.5 kB 9.1 MB/s eta 0:00:00


In [50]:
#@title Переменные, базовые функции
import requests
from datetime import datetime, timedelta
from google.colab import userdata
import openai
import json
import re
import os
from google.colab import userdata
from openai import OpenAI


token = userdata.get("PROXYAPI_KEY")
base_url = "https://api.proxyapi.ru/openai/v1"
client = OpenAI(
    api_key=token,
    base_url=base_url
)

# Создайте бесплатный api ключ для доступа к прогнозам погоды:
# https://home.openweathermap.org/users/sign_up
# https://home.openweathermap.org/api_keys
api_key = '08476c57274a8b387760da8c4d663cff' # Здесь нужно вставить ваш api ключ

# Функция прогноза погоды в городе на 5 дней вперед с интервалом 3 часа
# https://openweathermap.org/forecast5
def get_weather_forecast(city: str, dt, api_key=api_key):
    # city: str - город на русском
    # api_key: str - ключ https://home.openweathermap.org/api_keys
    url = 'http://api.openweathermap.org/data/2.5/forecast'
    params = {'lang': 'ru', 'q': city, 'units': 'metric', 'appid': api_key, 'limit': 5}
    forecasts = []
    try:
        r = requests.get(url=url, params=params)
        min_len = float("inf")
        ask_dt = datetime.strptime(dt, "%Y-%m-%d %H:%M").timestamp()
        for item in r.json()['list']:
          delta = abs(ask_dt - item['dt'] - 10800 * 2)
          if min_len > delta:
            min_len = delta
            continue
          else:
            day_time = ''
            time_forecast = datetime.fromtimestamp(item['dt']) + timedelta(hours=3) # UTC + 3 часа
            day_time += f"Город: {city}\n"
            day_time += f"Прогноз на {time_forecast}\n"
            day_time += f"Температура: {item['main']['temp']} C° {item['weather'][0]['description']}\n"
            day_time += f"Влажность: {item['main']['humidity']} %\n"
            day_time += f"Давление: {item['main']['pressure']} мм.рт.ст.\n"
            day_time += f"Ветер: {item['wind']['gust']} м/с\n"
            return day_time
        return forecasts # список прогнозов на пять дней вперед с интервалом в три часа
    except Exception as ex:
        print(ex)

In [51]:
#@title Решение
# Функция генерации ответа от OpenAI
def generate_answer(prompt_system, prompt_user, prompt_assistant='', model='gpt-4o-mini', temp=0.):
    # return '{"city": "Москва", "dt": "2026-03-28 18:00"}'
    messages = [
        {"role": "system", "content": prompt_system},
        {'role': 'assistant', 'content': prompt_assistant},
        {"role": "user", "content": prompt_user}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temp)
    return response.choices[0].message.content

# Функция согласования времени для записи на консультацию
def forecast(weather_request: str, model='gpt-4o-mini') -> str:
    # weather_request: str строка с запросом о погоде
    system_prompt = "Ты распознаватель места, даты и времени по заданному тексту"
    user_prompt = f"""Текущая дата и время: {(datetime.now() + timedelta(hours=3)).strftime("%Y-%m-%d %H:%M")}
    Опираясь на текущие дату и время, из запроса пользователя: {weather_request}
    - определи город и дату-время, про которые он спрашивает погоду.""" + """
    Результат выведи в формате json: {"city": "Город", "dt": "%Y-%m-%d %H:%M"}
    И не выводи ничего больше, кроме этого json
    """
    ask_place_dt = json.loads(generate_answer(system_prompt, user_prompt, model=model))
    # print(ask_place_dt)
    return get_weather_forecast(ask_place_dt["city"], ask_place_dt["dt"])

## Возможные варианты вопросов и ответов

In [52]:
# Текущая дата и время
# datetime.now() на сервере Colab - это UTC
print('Текущая дата и время:', (datetime.now() + timedelta(hours=3)).strftime("%Y-%m-%d %H:%M"))
# print(forecast('Дай прогноз погоды в Москве на послезавтра вечером'))

Текущая дата и время: 2026-03-26 22:53


In [53]:
print(forecast('Дай прогноз погоды в Москве на послезавтра вечером'))

Город: Москва
Прогноз на 2026-03-28 18:00:00
Температура: 10.43 C° пасмурно
Влажность: 65 %
Давление: 1016 мм.рт.ст.
Ветер: 6.15 м/с



In [54]:
print(forecast('Дай прогноз погоды в Барселоне через 3 дня днем'))

Город: Барселона
Прогноз на 2026-03-29 12:00:00
Температура: 11.77 C° облачно с прояснениями
Влажность: 26 %
Давление: 1018 мм.рт.ст.
Ветер: 5.46 м/с



In [55]:
print(forecast('Дай прогноз погоды в Париже через 4 дня вечером'))

Город: Париж
Прогноз на 2026-03-30 18:00:00
Температура: 11.98 C° небольшой дождь
Влажность: 56 %
Давление: 1026 мм.рт.ст.
Ветер: 9.57 м/с

